# Example: Let's Build a Ternary Commodity Price Tree
In this example, you will build a ternary price tree, i.e., a tree that assumes that the price of a good will go up, stay the same, or decrease tomorrow.

Fill me in.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

The [include command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

In [16]:
include("Include.jl");

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl), check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types and data used in this material. 

### Implementations
Fill me in

In [17]:
# Stars-and-bars helpers for recombining n-ary trees
stars_and_bars_level(n, k) = binomial(k + n - 1, n - 1);
total_nodes_T(h, n) = binomial(h + n, n);

## Example: Ternary Commodity Price Tree
In this example, we will build a ternary price tree, i.e., a tree that assumes that the price of a good will go up, stay the same, or decrease tomorrow.
To start, we'll setup some constants that we'll use, then we'll create a tree model, and then populate the data in the tree.

See the comment next to each constant for what it is, its permissible values, units, etc.

In [18]:
h = 2; # height of the tree {0,1,2} levels
n = 3; # branching factor of the tree, ternary = 3
price = 100.0; # initial price of the good
Δt = (1/365); # time step (1 day assuming 365 days per year)
u = exp(0.1*Δt); # up-factor (multiplicative)
d = exp(-0.1*Δt); # down-factor (multiplicative)
Δ = [u, 1.0, d]; # possible price changes, u or d

Ok, now we construct the recombining price tree using the constants we defined. We'll do this in two steps: 

* We first build the tree model with the specified height and branching factor, which creates the connectivity information; then we populate the tree with the price and path data. To build the tree model, we [use a `build(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/factory/#VLDataScienceMachineLearningPackage.build) and pass in the height h and branching factor n (as a NamedTuple). 
* We then pipe the resulting model [into the `populate!(...)` method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/factory/#VLDataScienceMachineLearningPackage.populate!) which updates the `data::Dict{Int64, NamedTuple}` field with price and path information. Notice that we use [the pipe `|>` operator](https://docs.julialang.org/en/v1/manual/functions/#Function-composition-and-piping) to pass the unpopulated model to [the `populate!(...)` method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/factory/#VLDataScienceMachineLearningPackage.populate!).

If you prefer code-first, you can skip ahead to the next cell and circle back to this explanation after running it once.

In [19]:
my_tree_model = build(MyAdjacencyRecombiningCommodityPriceTree, (
    h = h, # what is the height of the tree
    n = n, # what is the branching factor
)) |> m -> populate!(m, price, Δ); # wow! Fancy ...

### Are we doing what we expect?
Now that we have the `my_tree_model::MyAdjacencyRecombiningCommodityPriceTree` instance, let's verify that it has been constructed correctly.

In [20]:
@testset "Ternary Recombining Tree Verification" begin
    
    # Basic payload sanity
    @test all(length(nt.path) == n for nt in values(my_tree_model.data))
    @test all(all(p .>= 0) for nt in values(my_tree_model.data) for p in [nt.path])

    # Total node count matches theory: T(h, n) = binomial(h + n, n)
    total_nodes = length(my_tree_model.data)
    @test total_nodes == total_nodes_T(h, n)

    # Per-level node counts match L_k = binomial(k + n - 1, n - 1)
    counts = Dict{Int, Int}()
    for nt in values(my_tree_model.data)
        k = sum(nt.path)
        counts[k] = get(counts, k, 0) + 1
    end
    for k in 0:h
        expected = stars_and_bars_level(n, k)
        @test get(counts, k, 0) == expected
    end

    # Pricing formula verification: price = price₀ * ∏ Δ[j]^path[j]
    @testset "Pricing Formula Verification" begin
        for (idx, nt) in my_tree_model.data
            expected_price = price * prod(Δ .^ nt.path)
            @test isapprox(nt.price, expected_price; rtol = 1e-12, atol = 1e-10)
        end
    end
end;

Test Summary:                         | Pass  Total  Time
Ternary Recombining Tree Verification |   16     16  0.1s
Total  Time
Ternary Recombining Tree Verification |   16     16  0.1s


In [21]:
# (Optional) quick human-readable summary after tests
println("Ternary tree: h=$(h), n=$(n), total nodes=$(length(my_tree_model.data))")
for k in 0:h
    println("  level $k: ", sum(sum(nt.path) == k for nt in values(my_tree_model.data)))
end

Ternary tree: h=2, n=3, total nodes=10
  level 0: 1
  level 1: 3
  level 2: 6
